# Lab 6: Convolutional Neural Networks with PyTorch

In this lab we explore convolution filters through hands-on visual experiments, then build and train a Convolutional Neural Network (CNN) to classify images from the CIFAR-10 dataset. We compare performance with an equivalent fully connected network and study the effect of data augmentation.

In [ ]:
%pip install torch torchvision scipy

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.signal as signal

---
## Exercise 1: Convolution Filters and Feature Extraction

We take a colour photograph and study the effect of several hand-crafted convolution filters. The goal is to build intuition about what convolution layers learn in a CNN.

We use `scipy.signal.convolve` with `mode="same"` so the output image has the same spatial dimensions as the input.

In [ ]:
image = mpl.image.imread("./sheep.jpg")
plt.imshow(image)
print(image.shape)
print(image.dtype)

The image has three dimensions because it is in colour. The first two dimensions are height and width; the third dimension has size 3, one value per colour channel (red, green, blue). For example, the blue channel can be displayed as follows.

In [ ]:
plt.imshow(image[:, :, 2], cmap="Blues")

### Steps:

We define six convolution filters below. For each one:
1. Describe the visual effect of the filter on the image.
2. Explain *why* it has that effect based on the filter weights.

We add a third dimension to each filter with `[:, :, None]` so that `scipy` can broadcast it over all three colour channels.

**Filter 1**

In [ ]:
filter1 = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]])
filter1 = filter1[:, :, None]  # add a third dimension for scipy to apply it over a 3-channel image
conv = signal.convolve(image, filter1, mode="same")
print(conv.shape)
plt.imshow(conv)

**Filter 2**

In [ ]:
filter2 = np.array([[0, 0, 0], [0, .5, 0], [0, 0, 0]])
filter2 = filter2[:, :, None]
conv = signal.convolve(image, filter2, mode="same").astype(int)
plt.imshow(conv)

**Filter 3**

In [ ]:
filter3 = 1. / 25. * np.ones((5, 5))
filter3 = filter3[:, :, None]
conv = signal.convolve(image, filter3, mode="same").astype(int)
plt.imshow(conv)

**Filter 4**

In [ ]:
filter4 = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
filter4 = filter4[:, :, None]
conv = signal.convolve(image, filter4, mode="same")
plt.imshow(conv)

**Filter 5**

In [ ]:
filter5 = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]])
filter5 = filter5[:, :, None]
conv = signal.convolve(image, filter5, mode="same")
plt.imshow(conv)

**Filter 6 (Sobel filter)**

In [ ]:
filter6 = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])  # Sobel filter
filter6 = filter6[:, :, None]
conv = signal.convolve(image, filter6, mode="same")
plt.imshow(conv)

---
## Exercise 2: Image Classification with a CNN on CIFAR-10

We use the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset, which contains 50,000 colour images of size $32 \times 32$, each belonging to one of 10 classes: *airplane, car, bird, cat, deer, dog, frog, horse, ship, truck* (5,000 images per class).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from sklearn.model_selection import train_test_split

# Download CIFAR-10 (set download=False if files are already present)
dataset = torchvision.datasets.CIFAR10(root='./', train=True, download=True)

In [ ]:
X_dataset = dataset.data
y = torch.tensor(dataset.targets)
print(X_dataset.shape)
print(y.shape)

# Class names in the order they are stored in y
# (e.g. y[0] == 6 means the first image is a frog)
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
print(classes[y[0]])
plt.imshow(X_dataset[0].reshape(32, 32, 3))

### Part 2.1 – Data Preparation

PyTorch `Conv2d` expects tensors in **channels-first** format: `(N, C, H, W)`. The dataset is stored as `(N, H, W, C)`, so we need to permute the axes. We also normalise pixel values to $[-0.5, 0.5]$.

In [ ]:
# Permute to channels-first format: (N, H, W, C) → (N, C, H, W)
X = torch.tensor(X_dataset).permute(0, 3, 1, 2)
# Normalise to [-0.5, 0.5]
X = X.to(torch.float32) / 255. - 0.5
# We do NOT flatten X: the CNN will process it as a 2D image directly.

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.8)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, train_size=.5)

print(X_train.shape, y_train.shape)
print(X_val.shape,   y_val.shape)
print(X_test.shape,  y_test.shape)

### Part 2.2 – CNN Architecture

The following CNN (adapted from the [official PyTorch CIFAR-10 tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)) is proposed.

**Question 1.** What is the spatial size of a feature map (pay attention to the default padding of `Conv2d`):
- before the first convolution?
- after the first convolution?
- after the first MaxPool?
- after the second convolution?
- after the second MaxPool?

Is this consistent with the input size of the first `Linear` layer?

In [ ]:
def cnn():
    model = nn.Sequential(
        nn.Conv2d(3, 6, kernel_size=5),   # (N,  3, 32, 32) → (N,  6, 28, 28)
        nn.ReLU(),
        nn.MaxPool2d(2, stride=2),         # → (N,  6, 14, 14)
        nn.Conv2d(6, 16, kernel_size=5),   # → (N, 16, 10, 10)
        nn.ReLU(),
        nn.MaxPool2d(2, stride=2),         # → (N, 16,  5,  5)
        nn.Flatten(),                      # → (N, 16*5*5 = 400)
        nn.Linear(16 * 5 * 5, 120),
        nn.ReLU(),
        nn.Linear(120, 84),
        nn.ReLU(),
        nn.Linear(84, 10),
        nn.LogSoftmax(dim=1)               # compatible with nn.NLLLoss()
    )
    print(model)
    return model

model = cnn()
loss_function = nn.NLLLoss()

**Question 2.** Count the number of trainable parameters (weights + biases) for each layer. Then compare the total to the number of parameters a single `Linear` layer with 100 output neurons applied directly on the raw input would have.

In [ ]:
# Verify the total parameter count
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total:,}")

### Part 2.3 – Training the CNN

In [ ]:
def accuracy(model, X, y):
    """Returns the accuracy of `model` on dataset (X, y)."""
    y_pred = model(X)
    return (torch.argmax(y_pred, dim=1) == y).float().mean()


def train_cifar10(model, nb_epoch=5, batch_size=50, samples_used=None):
    samples_used = samples_used if samples_used else len(X_train)
    loss_train, loss_val = [], []

    optimizer = optim.Adam(model.parameters())
    for k in range(nb_epoch):
        for i in range(0, samples_used, batch_size):
            X_batch = X_train[i:i + batch_size]
            y_batch = y_train[i:i + batch_size]
            y_pred  = model(X_batch)
            loss    = loss_function(y_pred, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            loss_train.append(float(loss))
            loss_val.append(float(loss_function(model(X_val), y_val)))

        print(f"Epoch {k} done, loss = {loss:.4f}")

    with torch.no_grad():
        print("Training accuracy:", accuracy(model, X_train[:samples_used], y_train[:samples_used]).item())
        print("Validation accuracy:", accuracy(model, X_val, y_val).item())
    return loss_train, loss_val


def plot_losses(loss_train, loss_val):
    plt.figure()
    plt.plot(loss_train, label="Training loss")
    plt.plot(loss_val,   label="Validation loss")
    plt.legend()
    plt.grid()
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()

In [ ]:
%%time
model = cnn()
loss_train, loss_val = train_cifar10(model, nb_epoch=2)
with torch.no_grad():
    print("Test accuracy:", accuracy(model, X_test, y_test).item())

In [ ]:
plot_losses(loss_train, loss_val)

You should obtain around **50% accuracy** after 2 epochs. This may seem low, but it is far better than random chance (10%) and the task is significantly harder than MNIST. More complex architectures (e.g., ResNets) can exceed 95% on CIFAR-10, but their training time and architectural complexity are beyond the scope of this course.

After training, save the model weights so you can reload them without retraining.

In [ ]:
PATH = './cifar_net.pth'
torch.save(model.state_dict(), PATH)
# To reload later: model = cnn(); model.load_state_dict(torch.load(PATH))

### Part 2.4 – Comparison with a Fully Connected Network

**Question 3.** Build a non-convolutional network with a similar number of parameters to the CNN above. You can start with `nn.Flatten()` to avoid reshaping the input manually. Train it for 20 epochs and compare accuracy and training curves with the CNN. Explain the differences.

In [ ]:
%%time
# TODO: Build a non-convolutional network with a similar number of parameters (~62,000)
# to the CNN above. Start with nn.Flatten() to handle the image input.
# Train it for 20 epochs using train_cifar10 and compare accuracy and loss curves.
model_fc = ...

loss_train_fc, loss_val_fc = train_cifar10(model_fc, nb_epoch=20)
print("Validation accuracy:", accuracy(model_fc, X_val, y_val).item())
plot_losses(loss_train_fc, loss_val_fc)

---
## Exercise 3: Data Augmentation

A common technique to improve generalisation from a fixed dataset is **data augmentation**: applying random transformations to images on-the-fly during training (flips, rotations, crops…) that preserve the class label. This artificially increases dataset diversity without storing extra images.

We also show how to use PyTorch's `DataLoader`, which handles mini-batch creation and shuffling automatically — the recommended approach for real workflows.

### Part 3.1 – Per-class Accuracy Analysis

Before augmenting the data, it is useful to identify which classes the model struggles with, as this can inform which additional data to collect.

**Question 4.** For the CNN trained above, compute:
- **Recall per class:** for each class, what fraction of examples from that class are correctly classified?
- **False positive rate per class:** among all misclassified examples, what fraction are incorrectly assigned to each class?

In [ ]:
# Adapted from https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html

def evaluate_model(model):
    correct_pred = {c: 0 for c in classes}
    total_pred   = {c: 0 for c in classes}
    wrong_pred   = {c: 0 for c in classes}

    with torch.no_grad():
        y_pred = torch.argmax(model(X), dim=1)
        for i in range(X.shape[0]):
            # TODO: update correct_pred, total_pred, and wrong_pred
            # Hint: use classes[y[i]] for the true label and classes[y_pred[i]] for the predicted label
            ...

    # TODO: print recall per class
    # (for each class: fraction of examples of that class correctly classified)
    ...

    # TODO: print false positive rate per class
    # (for each class: fraction of all errors that were attributed to that class)
    ...


model = cnn()
model.load_state_dict(torch.load(PATH))
evaluate_model(model)

### Part 3.2 – Training with a DataLoader and Random Augmentations

PyTorch does not explicitly add new images to the dataset; instead, it applies random transformations *when images are loaded into each batch*, so every epoch the model sees slightly different versions of the same images.

In [ ]:
import torchvision.transforms.v2 as transforms

# Augmentation pipeline applied during training only
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset_aug = torchvision.datasets.CIFAR10(root='./', transform=transform)
train_set, val_set, test_set = torch.utils.data.random_split(dataset_aug, [40000, 5000, 5000])

# DataLoader handles mini-batch creation and optional shuffling
trainloader = torch.utils.data.DataLoader(train_set, batch_size=50, shuffle=True)
valloader   = torch.utils.data.DataLoader(val_set,   batch_size=100, shuffle=False)
testloader  = torch.utils.data.DataLoader(test_set,  batch_size=100, shuffle=False)

In [ ]:
%%time
model = cnn()
optimizer = optim.Adam(model.parameters())
nb_epoch = 5

train_loss_aug, val_loss_aug = [], []

for epoch in range(nb_epoch):
    model.train()
    for inputs, labels in trainloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

    train_loss_aug.append(loss.item())
    print(f"Epoch {epoch} done, loss = {loss.item():.4f}")

    # Validation loss (averaged over batches)
    model.eval()
    with torch.no_grad():
        val_loss_batch, nb_batches = 0.0, 0
        for inputs, labels in valloader:
            val_loss_batch += loss_function(model(inputs), labels).item()
            nb_batches += 1
        val_loss_aug.append(val_loss_batch / nb_batches)

plot_losses(train_loss_aug, val_loss_aug)

In [ ]:
correct, total = 0, 0
model.eval()
with torch.no_grad():
    for images, labels in valloader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy on the 5,000 validation images: {100 * correct // total} %")

**Question 5.** Compare the two training setups — without augmentation/shuffling vs. with augmentation and shuffled batches — on the same number of epochs. Comment on:
1. Training speed per epoch.
2. Smoothness of the loss curves.
3. Final validation accuracy.
4. Overfitting behaviour with more epochs (≥ 15).